# ARC-AGI-3 Kaggle Submission Notebook

Starter notebook con agente heurístico + memoria.

In [ ]:
!pip install -q arc-agi pandas numpy

In [ ]:
import json
import os
import random
from collections import defaultdict
import arc_agi


In [ ]:
class SubmissionAgent:
    def __init__(self, max_steps=100):
        self.max_steps = max_steps
        self.coord_candidates = [(0,0),(0,15),(15,0),(15,15),(3,3),(7,7),(8,4),(4,8),(10,10)]
        self.reset()

    def reset(self):
        self.step_count = 0
        self.visited_states = set()
        self.action_counts = defaultdict(int)
        self.action_success = defaultdict(float)
        self.last_reward = 0.0

    def state_key(self, obs):
        return repr(obs)[:1000]

    def act(self, env, obs):
        if obs is not None:
            self.visited_states.add(self.state_key(obs))

        actions = list(env.action_space)

        def action_score(action):
            name = str(action)
            success_bonus = self.action_success[name]
            usage_penalty = self.action_counts[name]
            return success_bonus - 0.25 * usage_penalty

        actions.sort(key=action_score, reverse=True)
        pool = actions[:max(1, len(actions)//2)]
        action = random.choice(pool)

        action_data = {}
        if hasattr(action, "is_complex") and action.is_complex():
            x, y = random.choice(self.coord_candidates)
            action_data = {"x": x, "y": y}

        self.action_counts[str(action)] += 1
        return action, action_data

    def update(self, action, reward):
        delta = reward - self.last_reward
        self.action_success[str(action)] += delta
        self.last_reward = reward


In [ ]:
arc = arc_agi.Arcade()
env = arc.make("ls20", render_mode=None)
agent = SubmissionAgent()

obs = None
reward = 0.0
terminated = False
truncated = False
info = {}
trace = []

for step in range(agent.max_steps):
    action, action_data = agent.act(env, obs)
    obs, reward, terminated, truncated, info = env.step(action, data=action_data)
    agent.update(action, reward)

    trace.append({
        "step": step,
        "action": str(action),
        "action_data": action_data,
        "reward": reward
    })

    if terminated or truncated:
        break


In [ ]:
submission = {
    "scorecard": str(arc.get_scorecard()),
    "trace": trace
}

os.makedirs("/kaggle/working", exist_ok=True)
with open("/kaggle/working/submission.json", "w") as f:
    json.dump(submission, f, indent=2)

print("Submission generada")
print(arc.get_scorecard())
